In [ ]:
import torch
from torch import nn
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
from rt_whisper.models import BoundaryWordFilter

In [ ]:
EXPONENT = 2
HEAD_MODEL = "/workspaces/dev/test/optimize/all/model_weight/head_model-pt-(m).pth"
TAIL_MODEL = "/workspaces/dev/test/optimize/all/model_weight/tail_model-pt-(m).pth"

In [ ]:
head_model_path = Path(HEAD_MODEL)
tail_model_path = Path(TAIL_MODEL)

if not head_model_path.exists():
    raise FileNotFoundError(f"Head model file not found: {head_model_path}")
if not tail_model_path.exists():
    raise FileNotFoundError(f"Tail model file not found: {tail_model_path}")

In [ ]:
head_model = BoundaryWordFilter.load(head_model_path)
tail_model = BoundaryWordFilter.load(tail_model_path)

In [ ]:
def generate_data(size:int, boundary:int = 16000, dur:int = None):
    for _ in range(size):
        start, end = sorted(np.random.randint(0, boundary, 2))
        if dur is not None:
            if start + dur <= boundary:
                end = start + dur
            else:
                start = end - dur

        start = start // 160 * 160
        end = end // 160 * 160
        mid = (start + end) / 2
        dur = end - start
        weight = (mid/boundary) ** EXPONENT

        start /= boundary
        end /= boundary
        mid /= boundary
        dur /= boundary

        yield (start, end, mid, dur), weight

def plot_mid_vs_pred_and_target(
    model: nn.Module,
    X: torch.Tensor,
    Y: torch.Tensor,
    device: str | torch.device = "cpu"
):
    model.to(device)
    X = X.to(device)
    Y = Y.to(device)

    with torch.no_grad():
        # mid만 추출 (X의 세 번째 컬럼)
        mids = X[:, 2].cpu().numpy()

        # 모델 예측
        preds = model(X).cpu().numpy().flatten()
        Y = Y.cpu().numpy().flatten()

    # 산점도 그리기
    plt.figure(figsize=(8,5))
    plt.scatter(mids, Y, alpha=0.5, label="Target (Y)", color="blue")
    plt.scatter(mids, preds, alpha=0.5, label="Model Output", color="red")
    plt.xlabel("mid")
    plt.ylabel("value")
    plt.title("Mid vs Model Prediction & Target")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.show()

In [ ]:
datasets = {dur:[data for data in generate_data(1024, dur=dur)] for dur in range(0, 16000, 1600)}

In [ ]:
def plot_with_dur(model:nn.Module, device: str|torch.device = "cpu"):
    for dur, data in datasets.items():
        print(f"========================= {dur} ===============================")
        X, Y = zip(*data)
        X = torch.tensor(X, dtype=torch.float32)
        Y = torch.tensor(Y, dtype=torch.float32).reshape(-1, 1)

        plot_mid_vs_pred_and_target(model, X, Y, device)

In [ ]:
plot_with_dur(head_model, "cuda")

In [ ]:
plot_with_dur(tail_model, "cuda")